# HulagwAI — model training (Google Colab)
**Before you run anything:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

Everything lives in the repo folder `ml-service/` (Python files in `training/`). This notebook only calls them, so the repo stays the single source of truth.
Full instructions: `ml-service/training/README.md`.

## 1 · Mount Google Drive (so work survives a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hulagwai_work'   # all outputs go here

## 2 · Get the code
Either clone your repo, or upload `ml-service.zip` to Drive and unzip it.

In [ ]:
# Option A - clone (replace the URL; for a private repo use a token: https://<TOKEN>@github.com/<you>/<repo>.git)
# !git clone https://github.com/<you>/<repo>.git /content/repo
# %cd /content/repo/ml-service

# Option B - from Drive
# !unzip -q /content/drive/MyDrive/ml-service.zip -d /content/repo
# %cd /content/repo/ml-service

## 3 · Install training dependencies

In [ ]:
!pip install -q -r requirements-train.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 4 · Kaggle access (for CIFAKE / ArtiFact)
Kaggle ▸ *Settings ▸ API ▸ Create New Token* gives you a username and key. In Colab open the 🔑 **Secrets** panel, add `KAGGLE_USERNAME` and `KAGGLE_KEY`, and tick *Notebook access*.

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 5 · (Optional) quick smoke test — 2-3 minutes
Proves the pipeline works before you spend hours. Uses tiny subsets and random weights, so the numbers mean nothing.

In [ ]:
# !python -m training.run_all --profile smoke --work /content/smoke --version 0.0.1

## 6 · Real training
`colab_fast` ≈ CIFAKE + ArtiFact (+ GenImage if you provide it). Allow roughly **2–4 hours** on a free T4 (estimate — depends on Colab load). Safe to re-run after a disconnect: finished stages are skipped.

Optional extras:
* `--genimage-root /content/drive/MyDrive/genimage` — a GenImage subset you downloaded (see `docs/DATASETS.md`)
* `--with-ood` — downloads **Synthbuster** (generators never seen in training) for the honest final test
* `--ood-real-dir <folder>` — real camera photos for that test (e.g. RAISE-1k)

In [ ]:
!python -m training.run_all --profile colab_fast --work $WORK --version 1.0.0 --with-ood

## 7 · Look at the result

In [ ]:
import json
m = json.load(open(f'{WORK}/reports/metrics.json'))
t = m['test']
print({k: round(t[k], 4) for k in ('accuracy','precision','recall','specificity','f1','roc_auc','mae','mse','brier','log_loss','ece')})
print('3-way verdict:', m['selective'])
print('robustness:', m['robustness'])
print('unseen generators:', m.get('ood'))

## 8 · Download the bundle
The zip is also in your Drive folder. Unzip it into **`ml-service/models/current/`** on your laptop.

In [ ]:
from google.colab import files
import glob
files.download(sorted(glob.glob(f'{WORK}/hulagwai-ensemble-*.zip'))[-1])